In [12]:

# !pip install pandas pyarrow scikit-learn seaborn matplotlib 

In [1]:
import pandas as pd
import pyarrow
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pickle


In [2]:
import mlflow

mlflow.set_tracking_uri('sqlite:///mlflow.db')
mlflow.set_experiment('nyx-taxi-experiment')

/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='/Users/dubai/MLOps_course/mlops_zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1786957730460, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786957730460, lifecycle_stage='active', name='nyx-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge


from sklearn.metrics import root_mean_squared_error

In [4]:
train_file = 'data/green_tripdata_2021-01.parquet'
val_file = 'data/green_tripdata_2021-02.parquet'
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime 
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds() / 60)

    df = df[((df.duration >=1) & (df.duration <=60))]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [5]:
df_train = read_dataframe(train_file)
df_val = read_dataframe(val_file)

print(len(df_train), len(df_val))

df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

73908 61921


In [6]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [7]:
lr = LinearRegression()

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_true=y_val, y_pred=y_pred)

7.480873453689756

In [8]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [11]:
with mlflow.start_run():
    mlflow.set_tag('developer', 'aigerim')
    mlflow.log_param('train-data-path', train_file)
    mlflow.log_param('valid-data-path', val_file)

    alpha = 0.01

    mlflow.log_param('alpha', alpha)

    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric('rmse', rmse)


In [9]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [10]:
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [14]:
def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [15]:
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:linear',
    'seed': 42
}

best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=5,
    trials=Trials()
)

[0]	validation-rmse:11.42373                         
  0%|          | 0/5 [00:00<?, ?trial/s, best loss=?]

/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:06:08] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[1]	validation-rmse:11.03631                         
[2]	validation-rmse:10.81856                         
[3]	validation-rmse:10.67044                         
[4]	validation-rmse:10.54971                         
[5]	validation-rmse:10.44404                         
[6]	validation-rmse:10.35622                         
[7]	validation-rmse:10.27498                         
[8]	validation-rmse:10.19718                         
[9]	validation-rmse:10.13349                         
[10]	validation-rmse:10.07418                        
[11]	validation-rmse:10.00913                        
[12]	validation-rmse:9.95462                         
[13]	validation-rmse:9.90772                         
[14]	validation-rmse:9.86436                         
[15]	validation-rmse:9.81709                         
[16]	validation-rmse:9.77484                         
[17]	validation-rmse:9.73403                         
[18]	validation-rmse:9.69744                         
[19]	validation-rmse:9.66231

/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:06:51] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:11.45629                                                  
[4]	validation-rmse:11.35209                                                  
[5]	validation-rmse:11.26601                                                  
[6]	validation-rmse:11.19766                                                  
[7]	validation-rmse:11.13934                                                  
[8]	validation-rmse:11.08589                                                  
[9]	validation-rmse:11.04122                                                  
[10]	validation-rmse:11.00024                                                 
[11]	validation-rmse:10.96052                                                 
[12]	validation-rmse:10.92418                                                 
[13]	validation-rmse:10.88784                                                 
[14]	validation-rmse:10.85645                                                 
[15]	validation-rmse:10.82899                       

...

[2]	validation-rmse:10.80778                                                  
[3]	validation-rmse:10.66230                                                  
[4]	validation-rmse:10.54734                                                  
[5]	validation-rmse:10.44237                                                  
[6]	validation-rmse:10.35513                                                  
[7]	validation-rmse:10.27974                                                  
[8]	validation-rmse:10.20842                                                  
[9]	validation-rmse:10.14309                                                  
[10]	validation-rmse:10.08344                                                 
[11]	validation-rmse:10.02851                                                 
[12]	validation-rmse:9.97641                                                  
[13]	validation-rmse:9.93264                                                  
[14]	validation-rmse:9.89195                        

/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:08:24] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[3]	validation-rmse:10.33712                                                  
[4]	validation-rmse:10.19402                                                  
[5]	validation-rmse:10.07712                                                  
[6]	validation-rmse:9.96802                                                   
[7]	validation-rmse:9.87800                                                   
[8]	validation-rmse:9.79172                                                   
[9]	validation-rmse:9.71283                                                   
[10]	validation-rmse:9.65772                                                  
[11]	validation-rmse:9.59339                                                  
[12]	validation-rmse:9.53903                                                  
[13]	validation-rmse:9.49345                                                  
[14]	validation-rmse:9.45884                                                  
[15]	validation-rmse:9.42465                        

/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:08:40] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()



[6]	validation-rmse:11.32999                                                  
[7]	validation-rmse:11.28328                                                  
[8]	validation-rmse:11.23386                                                  
[9]	validation-rmse:11.20111                                                  
[10]	validation-rmse:11.16075                                                 
[11]	validation-rmse:11.12956                                                 
[12]	validation-rmse:11.09616                                                 
[13]	validation-rmse:11.06494                                                 
[14]	validation-rmse:11.03267                                                 
[15]	validation-rmse:11.00298                                                 
[16]	validation-rmse:10.96981                                                 
[17]	validation-rmse:10.94203                                                 
[18]	validation-rmse:10.91522                       

In [17]:
mlflow.xgboost.autolog(disable=True)

with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.21522370937264673,
        'max_depth': 26,
        'min_child_weight': 4.107367764834471,
        'objective': 'reg:linear',
        'reg_alpha': 0.09099402339275377,
        'reg_lambda': 0.04294359447106556,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:11.93369
[1]	validation-rmse:11.73819
[2]	validation-rmse:11.60874


/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:11:07] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[3]	validation-rmse:11.51659
[4]	validation-rmse:11.44232
[5]	validation-rmse:11.38288
[6]	validation-rmse:11.32999
[7]	validation-rmse:11.28328
[8]	validation-rmse:11.23386
[9]	validation-rmse:11.20111
[10]	validation-rmse:11.16075
[11]	validation-rmse:11.12956
[12]	validation-rmse:11.09616
[13]	validation-rmse:11.06494
[14]	validation-rmse:11.03267
[15]	validation-rmse:11.00298
[16]	validation-rmse:10.96981
[17]	validation-rmse:10.94203
[18]	validation-rmse:10.91522
[19]	validation-rmse:10.89187
[20]	validation-rmse:10.86629
[21]	validation-rmse:10.85033
[22]	validation-rmse:10.82650
[23]	validation-rmse:10.80697
[24]	validation-rmse:10.78635
[25]	validation-rmse:10.76483
[26]	validation-rmse:10.74349
[27]	validation-rmse:10.72431
[28]	validation-rmse:10.70387
[29]	validation-rmse:10.68565
[30]	validation-rmse:10.66702
[31]	validation-rmse:10.64708
[32]	validation-rmse:10.63085
[33]	validation-rmse:10.61668
[34]	validation-rmse:10.59857
[35]	validation-rmse:10.58358
[36]	validation-r

2026/08/20 12:11:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
